# Chapter 2: Data Collection and Ingestion

This notebook complements Chapter 2 by treating ingestion as a model-quality and platform-traceability problem.

Use the retail ranking track for clickstream, orders, and annotation capture. Use the support-assistant track for documents, prompt templates, retrieval traces, tool outputs, and evaluation conversations.

## Topics Covered
- choosing ingestion modes based on source semantics rather than connector convenience
- validating raw records before they become model-facing assets
- preserving replay boundaries, schema or template versions, and lineage metadata
- deciding what prompt and conversation data should be retained, redacted, or quarantined

## Part 1: Batch Ingestion Example

Treat this batch example as a source-semantics exercise. Connector choice is secondary here. First decide whether the source is bounded or continuous, append-oriented or replacement-oriented, and easy or hard to replay. Only then decide whether a file drop, event stream, API pull, or version-controlled template store is the right transport.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def find_companion_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "datasets").exists() and (candidate / "code").exists():
            return candidate
    raise FileNotFoundError("Could not find companion repository root.")

NOTEBOOK_DIR = Path.cwd()
COMPANION_ROOT = find_companion_root(NOTEBOOK_DIR)
BASE_TIME = datetime(2026, 1, 15, 12, 0, 0)
print("Companion root located successfully.")

In [ ]:
# Simulate batch CSV ingestion
# Keep the source deterministic so the notebook reads like a repeatable contract.
def generate_batch_data(num_records=100):
    data = {
        'event_id': range(1, num_records + 1),
        'user_id': np.random.randint(1000, 2000, num_records),
        'event_type': np.random.choice(['click', 'view', 'purchase'], num_records),
        'timestamp': [BASE_TIME - timedelta(hours=i) for i in range(num_records)],
        'revenue': np.random.uniform(0, 100, num_records).round(2)
    }
    return pd.DataFrame(data)

np.random.seed(7)
batch_df = generate_batch_data()
source_contract = pd.DataFrame([
    {
        'source_id': 'retail_clickstream_batch',
        'capture_mode': 'batch_file_drop',
        'authoritative_or_observational': 'observational',
        'replay_boundary': 'daily_file_window',
        'schema_version': 'retail_events_v3',
        'downstream_asset': 'features_and_labels',
    },
    {
        'source_id': 'support_prompt_trace',
        'capture_mode': 'application_event_stream',
        'authoritative_or_observational': 'observational',
        'replay_boundary': 'request_trace_id',
        'schema_version': 'prompt_trace_v2',
        'downstream_asset': 'runtime_trace_and_evaluation',
    },
])
print(f"Ingested {len(batch_df)} records")
print(batch_df.head(5))
print('\nSource classification view:')
print(source_contract)

The code above is to provide one bounded source for asking the right questions in Chapter 2:

- What is the replay boundary?
- Which fields are authoritative, and which are only convenient?
- What metadata would have to survive if this feed later produced labels, features, or benchmark rows?
- Would this same source still be acceptable if it became a user-facing support-assistant input rather than an offline analytical feed?
- Which parts of the contract are semantic, and which parts are only transport?

A prompt template stored in Git, a user prompt emitted as an event, and a retrieved-context trace stored in logs all move through different transport paths. The deeper question is whether each one remains reconstructable and meaningful after ingestion.


## Part 1A: Classify Common Ingestion Sources

The chapter covers more than one connector shape, so the notebook should make those source classes visible. Use the table below to compare authoritative versus observational sources, bounded versus continuous capture, replay expectations, and sensitivity before deciding how each source should enter the platform.

In [ ]:
ingestion_dataset_dir = COMPANION_ROOT / "datasets" / "ingestion"
sample_files = {
    "cdc_orders": ingestion_dataset_dir / "sample_cdc_orders.csv",
    "event_stream": ingestion_dataset_dir / "sample_event_stream.csv",
    "api_extract": ingestion_dataset_dir / "sample_api_extract.csv",
    "annotation_tasks": ingestion_dataset_dir / "sample_annotation_tasks.csv",
}
sample_counts = {name: len(pd.read_csv(path)) for name, path in sample_files.items()}

source_classification = pd.DataFrame([
    {
        "source_id": "cdc_orders",
        "track": "retail",
        "source_type": "transaction_change_log",
        "capture_mode": "cdc",
        "authoritative_or_observational": "authoritative",
        "bounded_or_continuous": "continuous",
        "replay_required": "yes",
        "sensitive_content": "moderate",
        "downstream_asset": "labels_and_state_reconstruction",
        "sample_rows": sample_counts["cdc_orders"],
    },
    {
        "source_id": "event_stream",
        "track": "retail",
        "source_type": "behavioral_events",
        "capture_mode": "event_stream",
        "authoritative_or_observational": "observational",
        "bounded_or_continuous": "continuous",
        "replay_required": "yes",
        "sensitive_content": "moderate",
        "downstream_asset": "features_and_benchmarks",
        "sample_rows": sample_counts["event_stream"],
    },
    {
        "source_id": "api_extract",
        "track": "retail",
        "source_type": "partner_export",
        "capture_mode": "api_pull",
        "authoritative_or_observational": "authoritative",
        "bounded_or_continuous": "bounded_snapshot",
        "replay_required": "partial",
        "sensitive_content": "low",
        "downstream_asset": "catalog_and_enrichment_tables",
        "sample_rows": sample_counts["api_extract"],
    },
    {
        "source_id": "annotation_tasks",
        "track": "retail",
        "source_type": "human_review",
        "capture_mode": "annotation_workflow",
        "authoritative_or_observational": "authoritative",
        "bounded_or_continuous": "bounded_batches",
        "replay_required": "yes",
        "sensitive_content": "moderate",
        "downstream_asset": "labels_and_benchmark_rows",
        "sample_rows": sample_counts["annotation_tasks"],
    },
    {
        "source_id": "document_ingestion",
        "track": "assistant",
        "source_type": "document_corpus",
        "capture_mode": "batch_file_drop",
        "authoritative_or_observational": "authoritative",
        "bounded_or_continuous": "bounded_snapshots",
        "replay_required": "yes",
        "sensitive_content": "moderate",
        "downstream_asset": "chunks_embeddings_and_retrieval_state",
        "sample_rows": "snapshot scoped",
    },
    {
        "source_id": "prompt_tool_trace",
        "track": "assistant",
        "source_type": "runtime_trace",
        "capture_mode": "application_event_stream",
        "authoritative_or_observational": "observational",
        "bounded_or_continuous": "continuous",
        "replay_required": "partial",
        "sensitive_content": "high",
        "downstream_asset": "evaluation_and_incident_reconstruction",
        "sample_rows": "request scoped",
    },
])
source_classification

### What to Notice in the Source Classification Table

The table matters because it separates source semantics from transport mechanics. A CDC feed, an annotation task export, and a prompt-tool trace may all look like rows once they land, but they carry different authority, replay, and sensitivity obligations.

## Part 1B: Inspect Ingestion Contract Artifacts

The chapter names compact artifact files because ingestion discipline should survive outside the notebook. The snippets below show the shape of a source contract, a metadata sidecar, and the small helper module used to reason about capture semantics.

In [ ]:
schema_contract_path = COMPANION_ROOT / "artifacts" / "ingestion" / "schema_contract.yaml"
metadata_sidecar_path = COMPANION_ROOT / "artifacts" / "ingestion" / "metadata_sidecar.yaml"
helper_path = COMPANION_ROOT / "code" / "ingestion" / "metadata_contracts.py"

print("schema_contract.yaml")
print("-" * 60)
print("\n".join(schema_contract_path.read_text(encoding="utf-8").splitlines()[:18]))
print("\nmetadata_sidecar.yaml")
print("-" * 60)
print("\n".join(metadata_sidecar_path.read_text(encoding="utf-8").splitlines()[:18]))
print("\nmetadata_contracts.py")
print("-" * 60)
print("\n".join(helper_path.read_text(encoding="utf-8").splitlines()[:18]))

## Part 2: Data Quality Validation

Validation at ingestion is the minimum control plane for later reliability. A minimum viable ingestion discipline should answer these questions:

- Does a raw landing zone exist?
- Are event time and ingestion time both captured?
- Is the schema or template version recorded?
- Is the replay boundary explicit?
- Do validation failures go to quarantine?
- Is sensitive prompt or conversation data redacted or access-controlled?
- Can downstream consumers discover what they are reading?

A good incident to keep in mind: a support assistant may start answering incorrectly after a prompt-template update even when the model and vector index stay fixed. If the platform logged only the prompt version and not the assembled prompt instance, retrieved context, or tool outputs, the ingestion layer has already made later investigation harder.

In [ ]:
# Data quality checks at ingestion
def validate_ingested_data(df, source_id, capture_mode, schema_version, replay_boundary, sensitivity_class):
    issues = []
    quarantine_reasons = []

    null_counts = df.isnull().sum()
    if null_counts.any():
        issues.append(f"Null values found: {null_counts[null_counts > 0].to_dict()}")
        quarantine_reasons.append('missing_required_field')

    duplicates = df.duplicated().sum()
    if duplicates > 0:
        issues.append(f"Found {duplicates} duplicate rows")
        quarantine_reasons.append('duplicate_record')

    if df['revenue'].min() < 0:
        issues.append('Negative revenue values found')
        quarantine_reasons.append('invalid_business_range')

    validation_report = pd.DataFrame([
        {
            'source_id': source_id,
            'capture_mode': capture_mode,
            'event_time_present': bool(df['timestamp'].notnull().all()),
            'ingestion_time_present': True,
            'schema_version': schema_version,
            'replay_boundary': replay_boundary,
            'validation_status': 'pass' if not issues else 'fail',
            'quarantine_count': len(quarantine_reasons),
            'sensitivity_class': sensitivity_class,
            'lineage_ready': True,
        }
    ])

    return validation_report, issues, quarantine_reasons


def validate_prompt_trace(prompt_trace):
    checks = [
        ('prompt_instance_id', bool(prompt_trace.get('prompt_instance_id'))),
        ('template_version', bool(prompt_trace.get('template_version'))),
        ('retrieved_chunk_ids', bool(prompt_trace.get('retrieved_chunk_ids'))),
        ('tool_name', bool(prompt_trace.get('tool_name'))),
        ('tool_version', bool(prompt_trace.get('tool_version'))),
        ('sensitive_content_flag', 'sensitive_content_flag' in prompt_trace),
    ]
    trace_report = pd.DataFrame(
        [{'field': field, 'present': present} for field, present in checks]
    )
    quarantine = 'warn'
    if not trace_report['present'].all():
        quarantine = 'quarantine'
    return trace_report, quarantine

validation_report, issues, quarantine_reasons = validate_ingested_data(
    batch_df,
    source_id='retail_clickstream_batch',
    capture_mode='batch_file_drop',
    schema_version='retail_events_v3',
    replay_boundary='daily_file_window',
    sensitivity_class='internal',
)

prompt_trace_example = {
    'prompt_instance_id': 'pi_2026_01_15_0007',
    'template_version': 'support_answer_template_v7',
    'retrieved_chunk_ids': ['c18', 'c07'],
    'tool_name': 'order_status_lookup',
    'tool_version': 'v2',
    'sensitive_content_flag': True,
}
trace_report, trace_decision = validate_prompt_trace(prompt_trace_example)

print('Validation report:')
print(validation_report)
if issues:
    print('\nDetected issues:')
    for issue in issues:
        print(' -', issue)
else:
    print('\nDetected issues: none')
print(f"\nQuarantine decision for source batch: {'quarantine' if quarantine_reasons else 'publish'}")
print('\nPrompt and tool-trace validation report:')
print(trace_report)
print(f"\nPrompt-trace decision: {trace_decision}")


The validation output is useful only if it changes platform behavior. In Chapter 2 terms, a real ingestion boundary needs more than a pass or fail message:

- bad records should be quarantined rather than dropped silently;
- schema or template version should be recorded with the payload;
- replay should remain possible after the failure is discovered;
- prompt and conversation data may need redaction, access control, or shorter retention than ordinary events;
- downstream consumers should be able to tell whether they are reading raw source state, validated source state, or a repaired backfill.

The added prompt-trace report is there for the same reason. A support-assistant answer is only reconstructable if prompt instance, template version, retrieved chunks, tool identity, and sensitivity flags survive ingestion together.


## Exercises

1. Choose four sources across the two reference tracks and assign each one a capture mode, replay boundary, validation rule, and lineage field.
2. Extend the validation example with one rule that would matter for prompt or tool traces, not only classic event logs.
3. Write a minimum viable ingestion checklist for a support assistant that ingests documents, prompt templates, retrieved passages, tool outputs, and human feedback.